In [ ]:
import xarray as xr
import rioxarray as rxr
from pyproj import Transformer
import tensorflow as tf
import numpy as np
import glob
import re
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from utils import retrieve_url, str2time
from data_funcs import int2fstep

In [ ]:
static_files = ["hrrr_elevs.608.tif", "hrrr_lons.pkl", "hrrr_lats.pkl"]
base_url = "https://demo.openwfm.org/web/data/fmda/tif"
data_path = "."

In [ ]:
for file in static_files:
    retrieve_url(
        f"{base_url}/{file}",
        f"{data_path}/{file}"
    )    

In [ ]:
elev = xr.open_dataset("hrrr_elevs.608.tif")

In [ ]:
import sys
sys.path.append("..")
from utils import read_pkl

In [ ]:
lats = read_pkl("hrrr_lats.pkl")
lons = read_pkl("hrrr_lons.pkl")

In [ ]:
lats.shape

In [ ]:
elev

In [ ]:
elev.band_data

In [ ]:
plt.imshow(elev.isel(band=0).band_data)

In [ ]:
from utils import read_yml
from data_funcs import read_and_clean, combine_nested

In [ ]:
# Params used for data filtering
params_data = read_yml("../params_data.yaml") 
params_data.update({
    'hours': None,
    'max_intp_time': 10000,
    'zero_lag_threshold':10000
})
feats = ['Ed', 'Ew', 'rain', 'wind', 'solar', 'elev', 'lat', 'lon']

filename = "fmda_rocky_202403-05_f05.pkl"
file_paths = [f'../data/{filename}']
train = read_and_clean(file_paths, atm_source="HRRR", params_data = params_data, verbose=True)
# train = subset_by_features(train, params['features_list'])
train = combine_nested(train)

In [ ]:
train.keys()

In [ ]:
train['loc'].keys()

In [ ]:
import importlib
import data_funcs
importlib.reload(data_funcs)
from data_funcs import dict_to_df

In [ ]:
loc = dict_to_df(train['loc'])

In [ ]:
loc

In [ ]:
ds = rxr.open_rasterio("hrrr_elevs.608.tif")

In [ ]:
type(ds)

In [ ]:
from moisture_rnn_xarray import lonlat_to_xy, xy_to_grid, xr_to_lonlat

In [ ]:
x, y = lonlat_to_xy(loc.lon, loc.lat, ds.rio.crs)

In [ ]:
ds

In [ ]:
x.shape

In [ ]:
grid_x, grid_y = xy_to_grid(x, y, ds)

In [ ]:
np.max(np.abs(grid_x - loc.pixel_x))

In [ ]:
type(lats)

In [ ]:
type(lons)

In [ ]:
lonlat = xr_to_lonlat(ds)

In [ ]:
lons2 = lonlat[0]
lats2 = lonlat[1]

In [ ]:
np.max(np.abs(lons2-lons))

In [ ]:
loc

In [ ]:
np.max(np.abs(grid_x - loc.pixel_x))

In [ ]:
np.max(np.abs(lon-loc.lon))

In [ ]:
lons.shape

In [ ]:
lats.shape

In [ ]:
lons2.shape

In [ ]:
lats.shape

In [ ]:
lons[0,0]

In [ ]:
lons2[0,0]

In [ ]:
lats[0,0]

In [ ]:
lats2[0,0]

In [ ]:
import numpy as np
from pyproj import Transformer

# Assuming longitude and latitude are 2D arrays (output of xr_to_lonlat)
# and `xarray_obj` is your xarray object

# Get the CRS from the xarray object
crs_xarray = ds.rio.crs

# Define the transformer for lon/lat to x/y
transformer = Transformer.from_crs("EPSG:4326", crs_xarray, always_xy=True)

# Use the transformer to convert the 2D arrays of lon/lat back to x/y
x_back, y_back = transformer.transform(lons2, lats2)

# Original x and y from the xarray object
x_original = ds['x'].values
y_original = ds['y'].values

# Create a meshgrid of the original x and y
x_mesh, y_mesh = np.meshgrid(x_original, y_original)

# Compute the differences between the reconstructed and original x/y
x_diff = x_back - x_mesh
y_diff = y_back - y_mesh

# Print the maximum absolute differences for validation
print("Max absolute difference in x:", np.max(np.abs(x_diff)))
print("Max absolute difference in y:", np.max(np.abs(y_diff)))
